In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Utilizzo del workload

<span id="usage"></span>
L'utilizzo è una misura della quantità di tempo in cui il QPU è bloccato per il tuo workload, e viene calcolato in modo diverso a seconda della modalità di esecuzione che stai usando.

* L'utilizzo della sessione corrisponde al tempo reale (wall-clock) in cui la sessione è attiva. Consulta [Durata della sessione](/guides/run-jobs-session#session-length) per ulteriori informazioni sulle transizioni di stato della sessione.
* L'utilizzo del batch è la somma del tempo quantistico (il tempo impiegato dal complesso QPU per elaborare il tuo job) di tutti i job nel batch.
* L'utilizzo di un singolo job è il tempo quantistico utilizzato dal job durante l'elaborazione.

Tieni presente che i job falliti o annullati vengono conteggiati nel tuo utilizzo in determinate circostanze — consulta la sezione [Job falliti e annullati](#failed-job) per i dettagli.

Per gli utenti con piano a pagamento, l'utilizzo determina il costo del workload. Consulta [Gestisci i costi](/guides/manage-cost) per i dettagli.

<span id="failed-job"></span>
## Utilizzo per job falliti e annullati
Quando un job è fallito o annullato, l'utilizzo riportato è il seguente:

* Modalità job o batch: L'utilizzo riportato è il tempo in cui il QPU era bloccato per eseguire il tuo workload fino al momento in cui è fallito o è stato annullato. Pertanto, se il fallimento o l'annullamento si è verificato prima del blocco, l'utilizzo riportato è zero. Altrimenti, l'utilizzo riportato del workload è la quantità di utilizzo accumulata prima che il workload fallisse o venisse annullato. Di conseguenza, alcuni job falliti non compaiono nell'utilizzo riportato e altri sì.

* Modalità sessione: L'utilizzo riportato è il tempo reale (wall-clock) in cui la sessione è attiva, indipendentemente dal numero di job che falliscono o vengono annullati.

<span id="view-usage"></span>
## Interrogare l'utilizzo effettivo di un workload
Dopo che un workload è stato completato, esistono diversi modi per visualizzare il suo utilizzo effettivo:

- Esegui [`batch.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/batch#usage) o [`session.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/session#usage) in `qiskit-ibm-runtime` 0.30 o versioni successive.  Se stai usando una versione precedente di `qiskit-ibm-runtime` (>= 0.23 e < 0.30), l'utilizzo può comunque essere trovato in `session.details()["usage_time"]` e `batch.details()["usage_time"]`.
- Usa [`GET /sessions/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/sessions#tags__sessions__operations__GetSessionDetailsExtendedController_getSessionDetails) per vedere l'utilizzo di un batch o di una sessione specifici.
- Usa [`GET /jobs/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/jobs#tags__jobs__operations__GetJobByIdController_getJobById) per vedere l'utilizzo di un singolo job.

<span id="instance-usage"></span>
## Visualizzare l'utilizzo dell'istanza
Puoi visualizzare l'utilizzo di un'istanza nella pagina [Istanze](https://quantum.cloud.ibm.com/instances) oppure, per chi dispone delle autorizzazioni appropriate, nella pagina [Analytics](https://quantum.cloud.ibm.com/analytics). Tieni presente che le pagine potrebbero mostrare numeri di utilizzo diversi perché calcolano l'utilizzo in modo differente.

La pagina Istanze mostra l'utilizzo in tempo reale degli ultimi 28 giorni (a finestra mobile), fino all'ora corrente del giorno corrente. L'utilizzo nella pagina Analytics viene ricalcolato ogni ora e include gli ultimi 28 giorni completi; cioè, mostra l'utilizzo dalle 00:00 di 28 giorni fa ad oggi, all'inizio dell'ora.

## Stimare l'utilizzo prima di inviare un job
Sebbene ottenere una stima locale accurata sia complicato dalle operazioni aggiuntive eseguite per la soppressione e la mitigazione degli errori, puoi usare questa formula di base per ottenere un'approssimazione dell'utilizzo stimato:

`<per sub-job overhead> + (rep_delay + <circuit length>) * <num executions>`

- `<per sub-job overhead>` è un overhead di circa 2s per sub-job. Questo include operazioni come il caricamento del payload nell'elettronica di controllo. Il tuo job primitiva potrebbe essere suddiviso in più sub-job se è troppo grande perché il motore di esecuzione lo possa elaborare tutto in una volta.
- `rep_delay` è un'opzione [personalizzabile dall'utente](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2#rep_delay), e il valore predefinito è dato da `backend.default_rep_delay`, che è 250 microsecondi sulla maggior parte dei backend IBM Quantum. Tieni presente che abbassare `rep_delay` riduce il tempo di esecuzione totale del QPU, ma a scapito di un aumento del tasso di errore nella preparazione degli stati; consulta la guida [Esecuzione con tasso di ripetizione dinamico](/guides/repetition-rate-execution) per ulteriori informazioni.
- `<circuit length>` è la lunghezza totale delle istruzioni. Ogni istruzione richiede un tempo diverso sul QPU, quindi la lunghezza totale varia da circuito a circuito. Una misurazione, ad esempio, può richiedere 56 volte più tempo di un gate `x`. `backend.target[<instruction>][<qubit>].duration` può essere usato per trovare la durata esatta di ogni istruzione. Una lunghezza tipica del circuito è probabilmente compresa tra 50 e 100 microsecondi. Se stai usando tecniche di soppressione o mitigazione degli errori con le primitive, istruzioni aggiuntive potrebbero essere inserite nel tuo circuito, il che aumenterebbe la lunghezza totale del circuito.
    > **Note:** L'[opzione sperimentale `scheduler_timing`](/guides/visualize-circuit-timing) restituisce il tempo totale del circuito, ma questo NON è il tempo usato per la fatturazione.
- `<num executions>` è il numero totale di circuito moltiplicato per il numero di shot, dove i circuiti sono quelli generati dopo la trasmissione degli elementi PUB.
    - Se stai usando tecniche di mitigazione degli errori con le primitive, circuito aggiuntivi potrebbero essere eseguiti come parte del processo di mitigazione, il che aumenterebbe il numero totale di esecuzioni. Inoltre, le tecniche avanzate di mitigazione degli errori come PEA e PEC hanno un overhead molto più elevato perché richiedono l'esecuzione di circuito per l'apprendimento del rumore.
    - L'Estimator raggruppa gli osservabili che commutano qubit per qubit, il che riduce il numero di esecuzioni.

Se non stai usando tecniche avanzate di mitigazione degli errori né un `rep_delay` personalizzato, puoi usare `2+0.00035*<num executions>` come formula rapida.

## Passi successivi
> **Tip:** - Esamina questi suggerimenti: [Minimizza il tempo di esecuzione del job](minimize-time).
>     - Imposta il [Tempo massimo di esecuzione](max-execution-time).
>     - Scopri come eseguire il transpiling localmente nella sezione [Transpiler](./transpile/).
>     - Prova la guida [Confronta le impostazioni del Transpiler](/guides/circuit-transpilation-settings).